# 🚗 Notebook 01: Data Collection & Ingestion Pipeline
> **Project**: Cambodia Used Car Price Prediction  
> **Data Source**: Khmer24 Public API (`cars-for-sale` category)  
> **Objective**: Extract, validate, and persist raw listings into a versioned Parquet data lake for downstream ML modeling.

---

## 📌 Overview
This notebook demonstrates and executes the **Extract & Load (EL)** stage of the pipeline:
1. **Anti-Bot Client**: Bypassing Cloudflare TLS fingerprinting using `curl_cffi`.
2. **API Exploration**: Exploring Khmer24 taxonomy (categories & provincial locations).
3. **Payload Inspection & Parsing**: Extracting structured specs (year, fuel, mileage, transmission) from raw `highlight_specs`.
4. **NLP Brand/Model Parser**: Resolving English & Khmer script titles into canonical brands/models.
5. **Pipeline Execution**: Running the ingestion job with multi-day change tracking.
6. **Data Quality Audit**: Measuring data completeness and volume progress against the Phase 1 target (≥2,000 listings).

In [ ]:
# ── 1. Setup & Imports ─────────────────────────────────────────────────────────
import os
import sys
import json
import glob
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path when running from notebooks/ directory
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import core project modules
from src.config import (
    CORE_API_BASE,
    POSTS_API_BASE,
    RAW_DATA_DIR,
    TARGET_CATEGORY,
    TARGET_PROVINCE,
    MAX_PAGES,
    SCRAPE_MODE,
    get_daily_parquet_filename,
)
from src.client import Khmer24Client
from src.schemas import AdListingModel
from src.parsers import extract_brand_model, parse_mileage
from src.storage import save_to_parquet, save_sample_csv, load_all_parquet
from pipeline.extract_load import run as run_el_pipeline

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("✅ Setup complete. Project root:", project_root)

---  
## 🌐 2. Exploring Khmer24 API Taxonomy (Categories & Locations)
Khmer24 exposes core taxonomy endpoints for vehicle categories and Cambodian administrative provinces.

In [ ]:
# Connect to Khmer24 Core API
with Khmer24Client(lang="en") as client:
    categories = client.fetch_categories()
    provinces = client.fetch_locations(location_type="province")

print(f"Retrieved {len(categories)} top-level categories and {len(provinces)} provinces/locations.\n")

# Display automobile related categories
auto_cat = next((c for c in categories if 'car' in c.get('slug', '').lower() or 'auto' in c.get('slug', '').lower() or c.get('id') == 2), None)
print("Automobile Category Info:")
print(json.dumps(auto_cat or categories[:2], indent=2))

# Display sample provinces as DataFrame
df_provinces = pd.DataFrame(provinces)[['id', 'en_name', 'slug']].head(10)
display(df_provinces)

---  
## 🔍 3. Inspecting Raw API Payload vs Parsed Pydantic Schema
Let's fetch 1 page (30 items) with `fields="all"` to observe how raw nested JSON maps to our validated `AdListingModel`.

In [ ]:
# Fetch a single page from the category feed
with Khmer24Client(lang="en") as client:
    sample_listings = client.scrape_category_feed(category_slug="cars-for-sale", max_pages=1)

print(f"Successfully collected {len(sample_listings)} listings from page 1.\n")

# Inspect the first parsed AdListingModel record
first_item = sample_listings[0]
print("Sample Parsed Listing:")
print(f"  ID           : {first_item.listing_id}")
print(f"  Title        : {first_item.listing_title}")
print(f"  Price        : ${first_item.price:,.2f}" if first_item.price else "  Price        : None")
print(f"  Brand / Model: {first_item.vehicle_brand} {first_item.vehicle_model}")
print(f"  Year         : {first_item.vehicle_model_year}")
print(f"  Tax Type     : {first_item.vehicle_tax_type}")
print(f"  Condition    : {first_item.vehicle_condition}")
print(f"  Mileage      : {first_item.vehicle_mileage_km} km" if first_item.vehicle_mileage_km else "  Mileage      : None")
print(f"  Province     : {first_item.province}")
print(f"  Scraped At   : {first_item.scraped_at}")

---  
## 🏷️ 4. NLP Brand & Model Title Extraction Testing
Sellers on Khmer24 frequently write titles in Khmer or mix brand, model, year, and options together.  
Our parser [`extract_brand_model()`](file:///D:/ITC3_AMS_2025/I4_AMS_S2/Y4_Internship/Car_price_prediction/src/parsers.py) uses a 3-stage multilingual regex pipeline.

In [ ]:
test_titles = [
    "Toyota Prius 2010 Option 4 Solar ក្រដាសពន្ធ",
    "ឡានតូយ៉ូតា Camry 2007 ពណ៌ស ឡានស្អាត",
    "Ford Ranger Wildtrak 2022 Bi-Turbo Diesel",
    "RX350 2016 F-Sport full option",
    "Benz C300 2018 full option ស្លាកលេខ",
    "ឡានHonda Civic 2021 ម្ចាស់ដើម",
    "BYD Atto 3 2023 EV new condition",
    "Kia Morning 2015 for sale cheap",
]

results = []
for title in test_titles:
    brand, model = extract_brand_model(title)
    results.append({"Raw Title": title, "Extracted Brand": brand, "Extracted Model": model})

df_nlp = pd.DataFrame(results)
display(df_nlp)

---  
## 🚀 5. Executing the Data Ingestion Pipeline
Run the automated **Extract & Load** pipeline:  
- Discovers existing Parquets to track multi-day recurring listings vs new discoveries  
- Scrapes the active feed window up to `max_pages`  
- Saves daily versioned Parquet (`cars_YYYY-MM-DD_v01.parquet`) + sample CSV

In [ ]:
# Run the EL pipeline (e.g. 10 pages = 300 listings max per batch)
PAGES_TO_SCRAPE = 10

print(f"Starting EL pipeline scrape (max_pages={PAGES_TO_SCRAPE}, mode={SCRAPE_MODE})...")
collected_count = run_el_pipeline(max_pages=PAGES_TO_SCRAPE, scrape_mode="feed_window")

print(f"\n🎉 EL Pipeline finished. Total listings in this run: {collected_count:,}")

---  
## 💾 6. Inspecting the Cumulative Raw Data Lake
Load all historical Parquet files across `data/raw/` to analyze the complete raw data lake.

In [ ]:
# Load all raw parquet files
df_raw = load_all_parquet(RAW_DATA_DIR)

print(f"Total raw records in Data Lake: {len(df_raw):,} rows, {len(df_raw.columns)} columns")
print(f"Unique Listing IDs           : {df_raw['listing_id'].nunique():,}")
print("\nData Types and Missing Values:")
display(df_raw.info())

# Display sample of raw data
preview_cols = ['listing_id', 'vehicle_brand', 'vehicle_model', 'vehicle_model_year', 'price', 'vehicle_condition', 'province', 'scraped_at']
display(df_raw[[c for c in preview_cols if c in df_raw.columns]].head(10))

---  
## 📊 7. Data Quality & Completeness Audit
Measure feature coverage percentages and track progress toward Phase 1 volume requirements (≥2,000 listings with ≥1,500 priced).

In [ ]:
if len(df_raw) > 0:
    # Calculate completeness for key modeling features
    key_cols = ['price', 'vehicle_model_year', 'vehicle_brand', 'vehicle_model', 'vehicle_condition', 'vehicle_tax_type', 'vehicle_mileage_km', 'province']
    existing_key_cols = [c for c in key_cols if c in df_raw.columns]
    
    coverage = (df_raw[existing_key_cols].notna().mean() * 100).sort_values(ascending=False)
    
    # Plot Feature Completeness
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(coverage.index, coverage.values, color='#2b5c8f', edgecolor='black', alpha=0.85)
    ax.set_xlim(0, 105)
    ax.set_xlabel('Completeness Rate (%)', fontweight='bold')
    ax.set_title('Khmer24 Raw Data Completeness by Feature', fontweight='bold', fontsize=14)
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', fontsize=10, fontweight='bold')
        
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Phase 1 Target Checklist
    total_unique = df_raw['listing_id'].nunique()
    priced_unique = df_raw[df_raw['price'].notna()]['listing_id'].nunique()
    
    print("=" * 60)
    print("🎯 Phase 1 Gate Check Summary:")
    print(f"  1. Total Unique Listings : {total_unique:,} / 2,000 (Goal: >= 2,000)")
    print(f"  2. Priced Unique Listings: {priced_unique:,} / 1,500 (Goal: >= 1,500)")
    if total_unique >= 2000 and priced_unique >= 1500:
        print("  Status: ✅ Phase 1 Data Volume Requirements MET!")
    else:
        print("  Status: ⏳ Progressing toward Phase 1 volume targets.")
    print("=" * 60)

---  
## 🏁 Next Step
Now that raw listings are ingested, validated, and stored in the data lake:
👉 Proceed to **[`02_eda_exploration.ipynb`](file:///D:/ITC3_AMS_2025/I4_AMS_S2/Y4_Internship/Car_price_prediction/notebooks/02_eda_exploration.ipynb)** for deep statistical analysis, price depreciation modeling, and anomaly detection.